In [1]:
!pip install -q pdfplumber scikit-learn streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 56.5 MB/s eta 0:00:00


In [33]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayoubcherguelaine/company-documents-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'company-documents-dataset' dataset.
Path to dataset files: /kaggle/input/company-documents-dataset


In [34]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("cankatsrc/invoices")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'invoices' dataset.
Path to dataset files: /kaggle/input/invoices


In [35]:
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2")

In [36]:
import pandas as pd

# 1. Combine texts and labels into a master DataFrame
df_dataset = pd.DataFrame({
    'text': texts,
    'label': labels
})

# 2. Inspect Schema & Data Types
print("="*40)
print("📋 MASTER DATASET SCHEMA")
print("="*40)
print(df_dataset.dtypes)
print("\n")

# 3. Check Class Balance Distribution
print("="*40)
print("📊 CLASS DISTRIBUTION SUMMARY")
print("="*40)
print(df_dataset['label'].value_counts())
print("\n")

# 4. Preview First 5 Prepared Rows
print("="*40)
print("🔍 PREPARED DATASET SAMPLE PREVIEW")
print("="*40)
display(df_dataset.head())

📋 MASTER DATASET SCHEMA
text     object
label    object
dtype: object


📊 CLASS DISTRIBUTION SUMMARY
label
Invoice    2676
Bill       2676
Receipt      80
Name: count, dtype: int64


🔍 PREPARED DATASET SAMPLE PREVIEW


,text,label
0,Invoice\nOrder ID: 10725\nCustomer ID: FAMIA\n...,Invoice
1,Invoice\nOrder ID: 10507\nCustomer ID: ANTON\n...,Invoice
2,Invoice\nOrder ID: 10962\nCustomer ID: QUICK\n...,Invoice
3,Invoice\nOrder ID: 10276\nCustomer ID: TORTU\n...,Invoice
4,Invoice\nOrder ID: 10854\nCustomer ID: ERNSH\n...,Invoice


In [37]:
import pandas as pd

# 1. Pull your existing Bill and Invoice text rows from df_dataset (or cap them at 1,000 each for speed)
existing_bills = df_dataset[df_dataset['label'] == 'Bill']['text'].tolist()[:1000]
existing_invoices = df_dataset[df_dataset['label'] == 'Invoice']['text'].tolist()[:1000]

# 2. Generate clean synthetic Receipt samples to match the volume
synthetic_receipts = [
    f"store receipt thank you for shopping items purchased coffee sandwich snack item count subtotal tax paid cash change terminal auth code {i}"
    for i in range(1000)
]

# 3. Combine into a perfectly balanced master DataFrame
df_master = pd.DataFrame({
    'text': existing_bills + existing_invoices + synthetic_receipts,
    'label': ['Bill']*len(existing_bills) + ['Invoice']*len(existing_invoices) + ['Receipt']*len(synthetic_receipts)
})

print("⚖️ Perfectly Balanced Master Dataset Distribution:")
print(df_master['label'].value_counts())

# Ready variables for training
X = df_master['text']
y = df_master['label']

⚖️ Perfectly Balanced Master Dataset Distribution:
label
Bill       1000
Invoice    1000
Receipt    1000
Name: count, dtype: int64


In [38]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from catboost import CatBoostClassifier

# 1. Map string labels to integers
label_mapping = {'Bill': 0, 'Invoice': 1, 'Receipt': 2}
inv_mapping = {0: 'Bill', 1: 'Invoice', 2: 'Receipt'}
y_encoded = y.map(label_mapping)

# 2. Stratified Train / Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print(f"Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

# 3. TF-IDF Vectorization
print("\n⚙️ Vectorizing text features via TF-IDF...")
vectorizer = TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# 4. Train CatBoost
print("🚀 Training CatBoost MultiClass Classifier...")
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    verbose=50,
    random_seed=42
)
model.fit(X_train_vec, y_train)

# 5. Evaluate
y_pred = model.predict(X_test_vec)
if y_pred.ndim > 1:
    y_pred = y_pred.flatten()

target_names = ['Bill', 'Invoice', 'Receipt']
acc = accuracy_score(y_test, y_pred)

print("\n" + "="*50)
print(f"🎯 OVERALL MODEL ACCURACY: {acc * 100:.2f}%")
print("="*50)

print("\n📊 CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=target_names))

print("="*50)
print("🧩 CONFUSION MATRIX")
print("="*50)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in target_names],
    columns=[f"Pred {c}" for c in target_names]
)
print(cm_df)

# 6. Save Model Artifact
with open('model.pkl', 'wb') as f:
    pickle.dump({'vectorizer': vectorizer, 'model': model, 'mapping': inv_mapping}, f)

print("\n💾 Model successfully saved to model.pkl!")

Training Samples: 2400 | Testing Samples: 600

⚙️ Vectorizing text features via TF-IDF...
🚀 Training CatBoost MultiClass Classifier...
0:	learn: 1.0069491	total: 1.3s	remaining: 6m 28s
50:	learn: 0.4771053	total: 23.6s	remaining: 1m 55s
100:	learn: 0.4673104	total: 35.3s	remaining: 1m 9s
150:	learn: 0.4596259	total: 46.3s	remaining: 45.7s
200:	learn: 0.4527739	total: 56.8s	remaining: 28s
250:	learn: 0.4472793	total: 1m 6s	remaining: 12.9s
299:	learn: 0.4431421	total: 1m 16s	remaining: 0us

🎯 OVERALL MODEL ACCURACY: 41.50%

📊 CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Bill       0.12      0.12      0.12       200
     Invoice       0.12      0.12      0.12       200
     Receipt       1.00      1.00      1.00       200

    accuracy                           0.41       600
   macro avg       0.41      0.42      0.41       600
weighted avg       0.41      0.41      0.41       600

🧩 CONFUSION MATRIX
                Pred Bill  Pred Invoice  Pred R

In [39]:
%%writefile app.py
import streamlit as st
import pickle
import pdfplumber
import re

def clean_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$.,%/\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

@st.cache_resource
def load_artifacts():
    with open('model.pkl', 'rb') as f:
        artifact = pickle.load(f)
    return artifact['vectorizer'], artifact['model']

vectorizer, model = load_artifacts()

st.title("📄 PDF Document Classifier (CatBoost)")
st.write("Upload an Invoice, Bill, or Receipt PDF to extract text and classify.")

uploaded_file = st.file_uploader("Choose a PDF file", type=["pdf"])

if uploaded_file is not None:
    raw_text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                raw_text += t + "\n"

    if raw_text.strip():
        processed_text = clean_text(raw_text)

        st.subheader("Extracted & Preprocessed Text")
        st.text_area("Content", processed_text, height=180)

        # Transform and Predict
        features = vectorizer.transform([processed_text]).toarray()
        prediction = model.predict(features)[0]

        if isinstance(prediction, (list, np.ndarray)):
            prediction = prediction[0]

        st.success(f"**Predicted Category:** {prediction}")
    else:
        st.error("Could not extract text from this PDF.")

Overwriting app.py


In [47]:
%%writefile app.py
import streamlit as st
import pickle
import pdfplumber
import re
import numpy as np

def clean_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$.,%/\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

@st.cache_resource
def load_artifacts():
    with open('model.pkl', 'rb') as f:
        artifact = pickle.load(f)
    # Unpack the dictionary saved during training
    return artifact['vectorizer'], artifact['model'], artifact['mapping']

vectorizer, model, inv_mapping = load_artifacts()

st.title("📄 PDF Document Classifier")
st.write("Upload an Invoice, Bill, or Receipt PDF to classify its document type.")

uploaded_file = st.file_uploader("Choose a PDF file", type=["pdf"])

if uploaded_file is not None:
    raw_text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                raw_text += t + "\n"

    if raw_text.strip():
        processed_text = clean_text(raw_text)

        st.subheader("Extracted & Preprocessed Text Preview")
        st.text_area("Content", processed_text, height=150)

        # 1. Vectorize text
        features = vectorizer.transform([processed_text]).toarray()

        # 2. Predict numeric class
        pred_idx = model.predict(features)[0]
        if isinstance(pred_idx, (list, np.ndarray)):
            pred_idx = int(pred_idx[0])
        else:
            pred_idx = int(pred_idx)

        # 3. Map back to string label ('Bill', 'Invoice', 'Receipt')
        predicted_label = inv_mapping.get(pred_idx, "Unknown")

        st.success(f"**Predicted Category:** {predicted_label}")
    else:
        st.error("Could not extract text from this PDF.")

Overwriting app.py


In [41]:
!curl ipv4.icanhazip.com

34.12.0.16


In [44]:
!streamlit run app.py &> /dev/null &

In [45]:
import subprocess
from google.colab import userdata
from pyngrok import ngrok

# 1. Fetch token securely from Colab Secrets
NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Terminate old sessions
ngrok.kill()
subprocess.Popen(["pkill", "-f", "streamlit"])

# 3. Connect ngrok tunnel
public_url = ngrok.connect(8501)
print(f"\n🚀 Your Streamlit App is live at: {public_url}\n")

# 4. Launch Streamlit
!streamlit run app.py &> /dev/null &


🚀 Your Streamlit App is live at: NgrokTunnel: "https://dictate-dandelion-bulginess.ngrok-free.dev" -> "http://localhost:8501"



In [46]:
import subprocess
from pyngrok import ngrok

# 1. Kill any lingering streamlit or ngrok processes
!pkill -f streamlit
ngrok.kill()

# 2. Start Streamlit in the background pointing to your app.py
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])

# 3. Create a fresh public ngrok URL
public_url = ngrok.connect(8501)
print(f"🚀 Your New Live Streamlit URL: {public_url}")

🚀 Your New Live Streamlit URL: NgrokTunnel: "https://dictate-dandelion-bulginess.ngrok-free.dev" -> "http://localhost:8501"
